# Robust brain tumor segmentation with missing MRI modalities

STAT 420/626 course project. Training and evaluation notebook for the three arms of the proposal (September 25, 2026):

* **M0** trains on complete inputs only.
* **M1** adds a second view with one random modality removed (modality dropout).
* **M2** adds the complete to missing consistency term $\lambda \cdot \mathrm{KL}(\mathrm{sg}(p^{full}) \,\|\, p^{miss})$ on top of M1.

All arms share one 2.5D UNet (encoder 32, 64, 128, 256; bottleneck 512; instance norm; ReLU; four class softmax), AdamW (learning rate 0.0001, weight decay 0.00001), batch size 4, 32 slices per training patient per epoch with about half containing tumor, at most 80 epochs, validation every 5 epochs, early stopping after 5 checks without improvement.

**What this notebook adds to the proposal without adding models or data**

1. A raw data audit with planted defect tests: subject level grouping by the five digit subject ID (one subject can have several timepoints), content level duplicate detection, a modality identity check, label and header checks.
2. Metric code that must pass analytic tests and agree with an independent implementation (MONAI) before it may produce results.
3. Common random numbers: for a given seed, M0, M1 and M2 share the initialization, the slice order, the augmentation parameters and the dropped modality, so M2 versus M1 isolates the consistency term.
4. A precision analysis on validation data before the test set is opened, a test set lockbox, and a blind rehearsal of the analysis.
5. Inference that reports patient level and training run level uncertainty, plus mechanism diagnostics (manipulation check, volume bias, false positives, calibration).

**Stages**

1. Stage 0: configuration and environment record (anywhere).
2. Stage A: raw audit, subject grouping, split, slice cache (CPU, once).
3. Stage B: known answer tests for metrics, inputs and losses (CPU).
4. Stage C: $\lambda$ selection with tuning seed 0, validation data only (GPU).
5. Stage D: precision analysis from validation predictions (GPU, short).
6. Stage E: final runs, M0, M1, M2 for seeds 11, 22, 33 (GPU, resumable).
7. Stage F: analysis code frozen, lockbox written, one test evaluation (GPU).
8. Stage G: blind rehearsal, confirmatory analysis, diagnostics (CPU).

**How to run.** Keep `SMOKE_TEST = True` for the first run. It builds a synthetic phantom cohort with planted defects and known answers, runs every stage on CPU in a few minutes, and checks that the pipeline recovers those answers. Then set `SMOKE_TEST = False`, point `DATA_ROOT` at the folder that holds the BraTS 2023 GLI training zip or its extracted folder (plus the mapping xlsx), and run stage by stage. Every stage writes its outputs under `WORK_ROOT` and skips work that is already done, so a crashed or interrupted session can simply be rerun.

**Decisions this notebook makes where the proposal left room**

1. The patient unit is the five digit subject ID, so all timepoints of one subject stay in one partition. Cases whose individual anatomy matches another case under a different ID (possibly the same person) are merged into one group as well; exact and near duplicates are excluded.
2. Review flags never exclude a case automatically. Exclusions decided after looking at the overlays are recorded with their reason in `MANUAL_EXCLUDE` before the split is built.
3. Brain mask: voxels that are nonzero in any modality. Crop box: brain mask union tumor labels, plus one slice of margin in z, so 2.5D neighbors at the crop edge are the true (empty) neighbors.
4. Every slice is zero padded to one fixed canvas (largest crop rounded up to a multiple of 16) at training and at inference, so instance normalization always sees the same geometry.
5. One early stopping criterion for all arms: mean validation Dice over complete inputs and the four single missing conditions, over WT, TC and ET.
6. $\lambda$ is selected with tuning seed 0, never with seeds 11, 22 or 33. M1 at the tuning seed is trained as a reference but $\lambda = 0$ is not selectable, otherwise M2 could collapse into M1.
7. Empty mask conventions follow BraTS: both empty gives Dice 1 and HD95 0; exactly one empty gives Dice 0 and HD95 equal to the volume diagonal (373.13 mm for 240 by 240 by 155 at 1 mm).
8. HD95 is the larger of the two directed 95th percentile surface distances, with the surface defined as the mask minus its 6 connected erosion. This is the MONAI convention and is checked against MONAI in Stage B.
9. Probability maps for every patient, condition and run would take roughly 600 GB at fp16. The notebook saves 8 bit region probabilities for a small random subset of test patients (first seed only) and accumulates calibration statistics for everything.
10. Inference time and memory are reported as promised, but the three arms share one architecture, so the informative cost comparison is training time per step and total GPU hours.
11. In smoke test mode only, the network is narrower, epochs are fewer, three quarters of the sampled slices contain tumor and the learning rate is 0.001, so the tiny phantom model converges within minutes on CPU. The real run uses the proposal values.

## Stage 0. Configuration and environment

In [ ]:
import os, re, json, time, math, random, hashlib, inspect, marshal, platform, shutil, zipfile, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage, stats
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SMOKE_TEST = True          # True: synthetic phantom cohort on CPU. False: BraTS 2023 GLI training data.
DATA_ROOT = Path("data")   # folder holding the training zip (or the extracted folder) and the mapping xlsx
WORK_ROOT = Path("work")   # every output goes here; a local SSD is strongly preferred for the slice cache
UNLOCK_TEST = SMOKE_TEST   # real data: set True only after Stage D and after reading the lockbox section
UNBLIND = SMOKE_TEST       # real data: set True only after the blind rehearsal in Stage G looks right
EXPORT_NIFTI = True        # write test predictions as NIfTI so an independent evaluator can rescore them

In [ ]:
@dataclass(frozen=True)
class Config:
    # data
    zip_name: str = "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData.zip"
    mapping_name: str = "BraTS2023_2017_GLI_Mapping.xlsx"
    case_regex: str = r"BraTS-GLI-(\d{5})-(\d{3})"
    modalities: tuple = ("t1n", "t1c", "t2w", "t2f")   # proposal names: T1, T1ce, T2, FLAIR
    # split: blocks of 20 groups are assigned 14 train, 3 validation, 3 test (70/15/15)
    split_seed: int = 626
    split_pattern: tuple = (14, 3, 3)
    # model (proposal 3.3)
    features: tuple = (32, 64, 128, 256)
    bottleneck: int = 512
    # optimization (proposal 3.4)
    lr: float = 1e-4
    weight_decay: float = 1e-5
    batch_size: int = 4
    slices_per_patient: int = 32
    tumor_fraction: float = 0.5
    max_epochs: int = 80
    val_every: int = 5
    patience_checks: int = 5
    lambdas: tuple = (0.01, 0.1, 1.0)
    tuning_seed: int = 0
    seeds: tuple = (11, 22, 33)
    # spatial augmentation, identical for all arms
    p_flip: float = 0.5
    p_rotate: float = 0.5
    max_rotate_deg: float = 15.0
    p_scale: float = 0.5
    scale_range: tuple = (0.9, 1.1)
    # evaluation and statistics
    n_boot: int = 2000
    n_perm: int = 10000
    boot_seed: int = 2026
    ni_margin: float = 0.02
    far_fp_mm: float = 10.0
    min_et_ml: float = 0.05
    n_prob_cases: int = 10
    # engineering (use num_workers=0 when running the notebook on Windows or macOS)
    num_workers: int = 4
    io_threads: int = 4
    fingerprint_factor: int = 8
    eval_batch: int = 32
    log_every: int = 2500

CFG = Config()
if SMOKE_TEST:
    CFG = replace(CFG, features=(8, 16, 32, 64), bottleneck=128, lr=1e-3, slices_per_patient=16, tumor_fraction=0.75, max_epochs=18,
                  val_every=2, patience_checks=3, lambdas=(0.1, 1.0), seeds=(11, 22), n_boot=500, n_perm=2000,
                  num_workers=0, io_threads=2, fingerprint_factor=4, eval_batch=16, log_every=10**9, n_prob_cases=1)

WORK = WORK_ROOT / ("smoke" if SMOKE_TEST else "brats2023_gli")
DIRS = {k: WORK / k for k in ("audit", "cache", "runs", "eval", "analysis", "qc")}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE.type == "cuda" else None
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
REGIONS = ("WT", "TC", "ET")
CONDITIONS = ("complete",) + tuple(f"no_{m}" for m in CFG.modalities)
MISSING = CONDITIONS[1:]
ARMS = ("M0", "M1", "M2")


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=AMP_DTYPE is not None)


def stamp():
    return time.strftime("%Y%m%dT%H%M%S")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(text.encode()).hexdigest()


def dice_np(a, b):
    """Dice of two boolean masks. Both empty gives 1 (BraTS convention)."""
    sa, sb = int(np.count_nonzero(a)), int(np.count_nonzero(b))
    if sa == 0 and sb == 0:
        return 1.0
    return 2.0 * int(np.count_nonzero(np.logical_and(a, b))) / (sa + sb)


ENV = {"created": stamp(), "python": platform.python_version(), "platform": platform.platform(),
       "torch": torch.__version__, "numpy": np.__version__, "nibabel": nib.__version__, "device": str(DEVICE),
       "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None, "amp_dtype": str(AMP_DTYPE),
       "smoke_test": SMOKE_TEST, "config": asdict(CFG)}
(WORK / "environment.json").write_text(json.dumps(ENV, indent=1))
print({k: ENV[k] for k in ("python", "torch", "device", "gpu", "amp_dtype", "smoke_test")})

### Synthetic phantom cohort (smoke test only)

The phantom is built so that the right answers are known in advance.

1. **Contrast design.** The tumor core (necrosis plus enhancing tumor) is visible on t1n, t1c and t2f, and edema on t2w and t2f. Inside the core, enhancing tumor consists of scattered random patches covering about a fifth of the core, visible **only on t1c**; on every other modality it looks exactly like necrosis. Without t1c its location cannot be inferred, so the best a model can do is to call the whole core necrosis. A correctly wired model trained with modality dropout (M1, M2) must therefore show a large ET Dice drop when t1c is missing and almost none when t1n is missing. If the drop shows up for the wrong modality, the channel bookkeeping is wrong.
2. **Planted defects** that the audit must catch: a missing t2w file, NaN voxels, a legacy label 4, an exact duplicate under a new subject ID, a t1n and t1c swap, a t2w and t2f swap, and one subject with two timepoints that must stay in one partition.
3. The cohort is written as a zip plus a mapping xlsx with a site column, so the extraction and provenance code paths run too.

In [ ]:
def make_phantom_cohort(data_root, cfg, n_subjects=24, shape=(64, 64, 40), seed=1234):
    rng = np.random.default_rng(seed)
    data_root = Path(data_root)
    if data_root.exists():
        shutil.rmtree(data_root)
    staging = data_root / "staging"
    train_dir = staging / Path(cfg.zip_name).stem
    train_dir.mkdir(parents=True)
    X, Y, Z = shape
    gx, gy, gz = np.meshgrid(np.arange(X), np.arange(Y), np.arange(Z), indexing="ij")
    center = np.array([X / 2, Y / 2, Z / 2])
    affine = np.diag([-1.0, -1.0, 1.0, 1.0])
    # gray matter, white matter, ventricle, edema, necrosis, enhancing tumor
    contrast = {"t1n": (100, 118, 55, 92, 65, 65),
                "t1c": (100, 118, 55, 95, 60, 190),
                "t2w": (100, 85, 215, 165, 175, 175),
                "t2f": (100, 88, 35, 175, 130, 130)}

    def ellipsoid(c, r):
        return ((gx - c[0]) / r[0]) ** 2 + ((gy - c[1]) / r[1]) ** 2 + ((gz - c[2]) / r[2]) ** 2 <= 1.0

    def subject(with_et):
        c0 = center + rng.normal(0, 1.5, 3)
        brain = ellipsoid(c0, np.array([X * 0.40, Y * 0.43, Z * 0.42]) * rng.uniform(0.92, 1.05, 3))
        wm = ellipsoid(c0 + rng.normal(0, 1.0, 3), np.array([X * 0.27, Y * 0.30, Z * 0.28]) * rng.uniform(0.85, 1.10, 3)) & brain
        vent = ellipsoid(c0 + np.array([0.0, 2.0, 1.0]) + rng.normal(0, 1.5, 3),
                         np.array([X * 0.10, Y * 0.16, Z * 0.12]) * rng.uniform(0.7, 1.3, 3))
        side = rng.choice([-1.0, 1.0])
        tc = center + np.array([side * rng.uniform(0.10, 0.20) * X, rng.uniform(-0.15, 0.15) * Y, rng.uniform(-0.1, 0.1) * Z])
        r_ed = rng.uniform(9.0, 12.0)
        r_core = r_ed * rng.uniform(0.60, 0.75)
        dist = np.sqrt((gx - tc[0]) ** 2 + (gy - tc[1]) ** 2 + (1.2 * (gz - tc[2])) ** 2)
        seg = np.zeros(shape, np.uint8)
        seg[(dist <= r_ed) & brain] = 2
        core = (dist <= r_core) & brain
        seg[core] = 1
        if with_et:
            field = ndimage.gaussian_filter(rng.standard_normal(shape), 2.0)
            seg[core & (field > np.quantile(field[core], 0.8))] = 3   # scattered patches, about a fifth of the core
        vent &= seg == 0
        bias = 1.0 + 0.1 * np.sin(np.pi * gx / X + rng.uniform(0, np.pi)) * np.cos(np.pi * gy / Y + rng.uniform(0, np.pi))
        imgs = {}
        for m, (gm, w, v, e, n, t) in contrast.items():
            img = np.full(shape, float(gm))
            img[wm] = w
            img[vent] = v
            img[seg == 2] = e
            img[seg == 1] = n
            img[seg == 3] = t
            img = img * bias * rng.uniform(0.7, 1.4) + rng.normal(0.0, 6.0, shape)
            img[~brain] = 0.0
            imgs[m] = img.astype(np.float32)
        return imgs, seg

    def save_case(cid, imgs, seg, swap=None, drop=None, nan=False, legacy4=False):
        d = train_dir / cid
        d.mkdir()
        imgs, seg = dict(imgs), seg.copy()
        if swap:
            imgs[swap[0]], imgs[swap[1]] = imgs[swap[1]], imgs[swap[0]]
        if nan:
            imgs["t1n"] = imgs["t1n"].copy()
            imgs["t1n"][X // 2, Y // 2, Z // 2] = np.nan
        if legacy4:
            seg[seg == 3] = 4
        for m in cfg.modalities:
            if m != drop:
                nib.save(nib.Nifti1Image(imgs[m], affine), str(d / f"{cid}-{m}.nii.gz"))
        nib.save(nib.Nifti1Image(seg, affine), str(d / f"{cid}-seg.nii.gz"))

    no_et = {4, 10, 17, 21}
    planted = {"exclude": {}, "review": {}, "same_partition": [["BraTS-GLI-00003-000", "BraTS-GLI-00003-001"]]}
    made = {}
    for s in range(n_subjects):
        cid = f"BraTS-GLI-{s:05d}-000"
        imgs, seg = subject(with_et=s not in no_et)
        made[s] = (imgs, seg)
        kw = {}
        if s == 5:
            kw["drop"] = "t2w"
            planted["exclude"][cid] = "missing t2w file"
        if s == 7:
            kw["nan"] = True
            planted["exclude"][cid] = "non finite voxels"
        if s == 9:
            kw["legacy4"] = True
            planted["exclude"][cid] = "legacy label 4"
        if s == 11:
            imgs, seg = made[2]
            planted["exclude"][cid] = "exact duplicate of subject 00002"
        if s == 13:
            kw["swap"] = ("t1n", "t1c")
            planted["review"][cid] = "t1n and t1c swapped"
        if s == 15:
            kw["swap"] = ("t2w", "t2f")
            planted["review"][cid] = "t2w and t2f swapped"
        save_case(cid, imgs, seg, **kw)
    save_case("BraTS-GLI-00003-001", *subject(with_et=True))   # second timepoint of subject 00003

    with zipfile.ZipFile(data_root / cfg.zip_name, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(train_dir.rglob("*.nii.gz")):
            zf.write(f, f.relative_to(staging))
    names = sorted(p.name for p in train_dir.iterdir())
    pd.DataFrame({"BraTS2023": names, "Site No": rng.integers(1, 4, len(names))}).to_excel(
        data_root / cfg.mapping_name, index=False)
    shutil.rmtree(staging)
    return planted


PHANTOM = None
if SMOKE_TEST:
    DATA_ROOT = WORK / "phantom_data"
    if not (DIRS["audit"] / "split.csv").exists():
        PHANTOM = make_phantom_cohort(DATA_ROOT, CFG)
        (WORK / "phantom_planted.json").write_text(json.dumps(PHANTOM, indent=1))
    PHANTOM = json.loads((WORK / "phantom_planted.json").read_text())
    print(json.dumps(PHANTOM, indent=1))

## Stage A. Raw audit, subject grouping, split and slice cache

One pass over the raw files. Each case is checked (files present, readable, identical shapes and affines, finite values, integer labels in {0, 1, 2, 3}), normalized per modality with its own nonzero voxels, cropped and cached. Hard failures are excluded with a logged reason. Soft findings go to a review list with overlays for visual inspection; they are never excluded automatically, because unusual tumor size or appearance is not an exclusion criterion.

In [ ]:
HARD, OK = "exclude", "ok"


def locate_training_data(data_root, cfg):
    data_root = Path(data_root)
    pattern = re.compile(cfg.case_regex)

    def case_dirs(base):
        return sorted(p for p in Path(base).rglob("*") if p.is_dir() and pattern.fullmatch(p.name))

    cases = case_dirs(data_root) if data_root.exists() else []
    if not cases:
        zips = sorted(data_root.rglob(cfg.zip_name))
        if not zips:
            raise FileNotFoundError(f"no case folders and no {cfg.zip_name} under {data_root.resolve()}")
        dest = zips[0].parent / "extracted"
        print(f"extracting {zips[0]} into {dest}")
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(dest)
        cases = case_dirs(dest)
    mapping = sorted(data_root.rglob(cfg.mapping_name))
    print(f"found {len(cases)} case folders; mapping file: {mapping[0] if mapping else 'not found'}")
    return cases, (mapping[0] if mapping else None)


def process_case(case_dir, cfg, cache_dir):
    """Audit one case from its raw files; on success write the normalized crop to the cache."""
    cid = case_dir.name
    subject_id, timepoint = re.fullmatch(cfg.case_regex, cid).groups()
    rec = {"case_id": cid, "subject_id": subject_id, "timepoint": int(timepoint), "status": OK, "reasons": ""}

    def fail(reason):
        rec.update(status=HARD, reasons=reason)
        return rec, None

    paths = {}
    for k in (*cfg.modalities, "seg"):
        hits = sorted(case_dir.glob(f"{cid}-{k}.nii*"))
        if not hits:
            return fail(f"missing {k} file")
        paths[k] = hits[0]
    try:
        imgs = {k: nib.load(str(p)) for k, p in paths.items()}
        arr = {k: np.asarray(im.get_fdata(dtype=np.float32)) for k, im in imgs.items()}
    except Exception as exc:
        return fail(f"unreadable file ({type(exc).__name__})")
    ref = imgs["seg"]
    if len({im.shape for im in imgs.values()}) != 1:
        return fail("shape differs across files")
    if not all(np.allclose(im.affine, ref.affine, atol=1e-3) for im in imgs.values()):
        return fail("affine differs across files")
    zooms = tuple(float(z) for z in ref.header.get_zooms()[:3])
    rec.update(shape="x".join(map(str, ref.shape)), orientation="".join(nib.aff2axcodes(ref.affine)),
               spacing="x".join(f"{z:g}" for z in zooms), image_dtype=str(imgs[cfg.modalities[0]].get_data_dtype()))
    for k, a in arr.items():
        bad = int(a.size - np.count_nonzero(np.isfinite(a)))
        if bad:
            return fail(f"{bad} non finite voxels in {k}")
    seg_f = arr.pop("seg")
    if not np.array_equal(seg_f, np.round(seg_f)):
        return fail("non integer label values")
    seg = seg_f.astype(np.int16)
    labels = sorted(int(v) for v in np.unique(seg))
    rec["labels"] = " ".join(map(str, labels))
    if not set(labels) <= {0, 1, 2, 3}:
        return fail(f"unexpected label values {labels}")

    mods = cfg.modalities
    nz = [arr[m] != 0 for m in mods]
    brain = np.logical_or.reduce(nz)
    tumor = seg > 0
    voxel_ml = float(np.prod(zooms)) / 1000.0
    rec.update(brain_voxels=int(brain.sum()), tumor_voxels=int(tumor.sum()),
               outside_brain_frac=float(np.count_nonzero(tumor & ~brain) / max(1, np.count_nonzero(tumor))),
               et_present=bool((seg == 3).any()), wt_components=int(ndimage.label(tumor, np.ones((3, 3, 3)))[1]))
    for name, lab in (("ncr", 1), ("ed", 2), ("et", 3)):
        rec[f"vol_{name}_ml"] = float(np.count_nonzero(seg == lab) * voxel_ml)
    for i, m in enumerate(mods):
        rec[f"mask_dice_{m}"] = dice_np(nz[i], brain)
        rec[f"negative_frac_{m}"] = float(np.count_nonzero(arr[m] < 0) / max(1, np.count_nonzero(nz[i])))

    # per volume, per modality z score over that modality's own nonzero voxels; background stays exactly zero
    z = np.zeros((len(mods),) + ref.shape, np.float32)
    for i, m in enumerate(mods):
        v = arr[m][nz[i]]
        z[i][nz[i]] = (v - float(v.mean())) / max(float(v.std()), 1e-6)
        rec[f"zmax_{m}"] = float(np.abs(z[i]).max())
    if rec["et_present"] and (seg == 2).any():
        for i, m in enumerate(mods):
            rec[f"et_contrast_{m}"] = float(z[i][seg == 3].mean() - z[i][seg == 2].mean())

    keep = brain | tumor

    def extent(axis):
        idx = np.where(keep.any(axis=tuple(a for a in range(3) if a != axis)))[0]
        return int(idx[0]), int(idx[-1]) + 1

    (x0, x1), (y0, y1), (z0, z1) = extent(0), extent(1), extent(2)
    z0, z1 = max(0, z0 - 1), min(ref.shape[2], z1 + 1)
    img_c = np.ascontiguousarray(z[:, x0:x1, y0:y1, z0:z1].transpose(3, 0, 1, 2).astype(np.float16))  # Z, C, X, Y
    seg_c = np.ascontiguousarray(seg[x0:x1, y0:y1, z0:z1].transpose(2, 0, 1).astype(np.uint8))       # Z, X, Y
    out = Path(cache_dir) / cid
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "img.npy", img_c)
    np.save(out / "seg.npy", seg_c)
    meta = {"case_id": cid, "shape": list(ref.shape), "spacing": list(zooms), "affine": ref.affine.tolist(),
            "bbox": [[x0, x1], [y0, y1], [z0, z1]],
            "brain_slices": np.where(brain[x0:x1, y0:y1, z0:z1].any(axis=(0, 1)))[0].tolist(),
            "tumor_slices": np.where(seg_c.reshape(seg_c.shape[0], -1).any(1))[0].tolist()}
    (out / "meta.json").write_text(json.dumps(meta))
    rec["crop_shape"] = "x".join(map(str, img_c.shape))

    f = cfg.fingerprint_factor

    def pool(a):
        s = [d // f * f for d in a.shape]
        a = a[: s[0], : s[1], : s[2]]
        return a.reshape(s[0] // f, f, s[1] // f, f, s[2] // f, f).mean(axis=(1, 3, 5))

    fp = {"img": np.stack([pool(z[i]) for i in range(len(mods))]).astype(np.float32),
          "wt": pool(tumor.astype(np.float32)).astype(np.float32)}
    rec["seg_sha256"] = hashlib.sha256(np.ascontiguousarray(seg.astype(np.uint8)).tobytes()).hexdigest()
    return rec, fp


def run_audit(case_dirs, cfg, cache_dir):
    recs, fps, t0 = [], {}, time.time()
    with ThreadPoolExecutor(max_workers=cfg.io_threads) as ex:
        for k, (rec, fp) in enumerate(ex.map(lambda d: process_case(d, cfg, cache_dir), case_dirs), 1):
            recs.append(rec)
            if fp is not None:
                fps[rec["case_id"]] = fp
            if k % 100 == 0 or k == len(case_dirs):
                print(f"  audited {k}/{len(case_dirs)} cases in {time.time() - t0:.0f} s")
    return pd.DataFrame(recs), fps

Cohort level checks use small pooled fingerprints of every case.

1. **Duplicates and hidden repeat subjects.** Identical label maps are exact duplicates. For everything else the notebook compares residual images (case minus the cohort median template, so the shared atlas anatomy cancels and individual anatomy remains). A residual correlation above 0.8 suggests the same individual, for example one person under two subject IDs; such pairs are flagged for review and merged into one group so they cannot straddle partitions. Pairs above 0.98 whose pooled tumor masks also overlap above 0.9 are treated as duplicates and excluded.
2. **Modality identity.** All BraTS images live in one atlas space, so each modality has a meaningful median template. A t2w channel that correlates best with the FLAIR template (or the reverse) is flagged. T1 versus T1ce cannot be told apart this way, so a second check compares enhancement: inside enhancing tumor, the contrast with edema must be larger on t1c than on t1n.

In [ ]:
class UnionFind:
    def __init__(self, items):
        self.parent = {i: i for i in items}

    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


FAMILY = {"t1n": "T1", "t1c": "T1", "t2w": "T2", "t2f": "FLAIR"}


def cohort_checks(audit, fps, cfg):
    audit = audit.copy()
    ids = sorted(c for c in audit.loc[audit.status != HARD, "case_id"] if c in fps)
    a = audit.set_index("case_id")
    review = {c: [] for c in ids}
    excluded = {}
    for _, grp in a.loc[ids].groupby("seg_sha256"):
        names = sorted(grp.index)
        for other in names[1:]:
            excluded[other] = f"exact duplicate of {names[0]}"

    img = np.stack([fps[c]["img"] for c in ids])                       # n, modality, x, y, z
    template = np.median(img, axis=0)
    R = (img - template).reshape(len(ids), -1)
    R = (R - R.mean(1, keepdims=True)) / (R.std(1, keepdims=True) + 1e-6)
    corr = R @ R.T / R.shape[1]
    W = np.stack([fps[c]["wt"].ravel() > 0.25 for c in ids]).astype(np.float32)
    size = W.sum(1)
    wt_dice = 2 * (W @ W.T) / np.maximum(size[:, None] + size[None, :], 1)
    ii, jj = np.where(np.triu(corr > 0.8, k=1))   # residual anatomy this similar suggests the same person
    pairs = pd.DataFrame({"case_a": [ids[i] for i in ii], "case_b": [ids[j] for j in jj],
                          "residual_corr": corr[ii, jj], "wt_dice": wt_dice[ii, jj]})
    pairs["same_subject"] = [a.at[p, "subject_id"] == a.at[q, "subject_id"] for p, q in zip(pairs.case_a, pairs.case_b)]

    uf = UnionFind(ids)
    for _, members in a.loc[ids].groupby("subject_id").groups.items():
        members = sorted(members)
        for m in members[1:]:
            uf.union(members[0], m)
    for p in pairs.itertuples():
        uf.union(p.case_a, p.case_b)
        if p.same_subject or p.case_b in excluded:
            continue
        if p.residual_corr > 0.98 and p.wt_dice > 0.9:
            excluded[p.case_b] = f"near duplicate of {p.case_a}"
        else:
            review[p.case_a].append(f"same anatomy as {p.case_b}? (residual correlation {p.residual_corr:.2f})")
            review[p.case_b].append(f"same anatomy as {p.case_a}? (residual correlation {p.residual_corr:.2f})")

    if all(m in FAMILY for m in cfg.modalities):
        fams = sorted(set(FAMILY.values()))
        fam_t = {f: np.mean([template[i] for i, m in enumerate(cfg.modalities) if FAMILY[m] == f], axis=0) for f in fams}
        mask = (template != 0).any(0)

        for k, c in enumerate(ids):
            healthy = mask & (fps[c]["wt"] == 0)          # compare tissue away from the tumor only
            if healthy.sum() < 20:
                continue
            for i, m in enumerate(cfg.modalities):
                score = {f: float(np.corrcoef(img[k, i][healthy], fam_t[f][healthy])[0, 1]) for f in fams}
                best = max(score, key=score.get)
                if best != FAMILY[m] and score[best] - score[FAMILY[m]] > 0.05:
                    review[c].append(f"{m} looks like {best} (template correlation "
                                     f"{score[best]:.2f} vs {score[FAMILY[m]]:.2f})")
    if {"et_contrast_t1n", "et_contrast_t1c"} <= set(a.columns):
        for c in ids:
            n_, c_ = a.at[c, "et_contrast_t1n"], a.at[c, "et_contrast_t1c"]
            if pd.notna(n_) and pd.notna(c_) and n_ >= c_:
                review[c].append("enhancement stronger on t1n than on t1c (possible swap)")

    modal = {k: a.loc[ids, k].mode().iat[0] for k in ("shape", "orientation", "spacing")}
    for c in ids:
        r = a.loc[c]
        for k, v in modal.items():
            if r[k] != v:
                review[c].append(f"{k} {r[k]} differs from cohort {v}")
        if r.tumor_voxels == 0:
            review[c].append("empty segmentation")
        if r.outside_brain_frac > 0.01:
            review[c].append(f"{100 * r.outside_brain_frac:.1f}% of tumor labels outside the brain mask")
        if min(r[f"mask_dice_{m}"] for m in cfg.modalities) < 0.95:
            review[c].append("brain masks disagree across modalities")

    for c, why in excluded.items():
        audit.loc[audit.case_id == c, ["status", "reasons"]] = [HARD, why]
    audit["review"] = audit.case_id.map(lambda c: "; ".join(review.get(c, [])))
    audit["group"] = audit.case_id.map(lambda c: uf.find(c) if c in uf.parent else "")
    return audit, pairs, modal


def read_site_table(path, cfg):
    """The 2023 GLI mapping file reportedly carries a site number per case. Column names are not assumed:
    the case ID column is found by pattern and the site column by name."""
    if path is None:
        print("mapping file not found; stratifying by ET presence only")
        return None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sheets = pd.read_excel(path, sheet_name=None)
    for name, df in sheets.items():
        df.columns = [str(c) for c in df.columns]
        print(f"mapping sheet {name!r}: {len(df)} rows, columns {list(df.columns)}")
        id_col = next((c for c in df.columns if df[c].astype(str).str.fullmatch(cfg.case_regex).mean() > 0.5), None)
        site_col = next((c for c in df.columns if "site" in c.lower()), None)
        if id_col is not None and site_col is not None:
            out = df[[id_col, site_col]].dropna().rename(columns={id_col: "case_id", site_col: "site"})
            out["site"] = out["site"].astype(str).str.replace(r"\.0$", "", regex=True)
            print(f"using {id_col!r} as case ID and {site_col!r} as site ({out.site.nunique()} sites)")
            return out.drop_duplicates("case_id")
    print("no usable case ID and site columns; stratifying by ET presence only")
    return None


def make_split(audit, sites, cfg):
    df = audit[audit.status != HARD].copy()
    df["site"] = df.case_id.map(sites.set_index("case_id")["site"]) if sites is not None else None
    df["site"] = df["site"].fillna("unknown")
    g = df.groupby("group").agg(et=("et_present", "max"), site=("site", lambda s: s.mode().iat[0]))
    et_tag = g["et"].map({True: "ET", False: "noET"})
    g["stratum"] = et_tag + "|" + g["site"]
    block = sum(cfg.split_pattern)
    small = g["stratum"].map(g["stratum"].value_counts()) < block
    g.loc[small, "stratum"] = et_tag[small] + "|pooled"
    rng = np.random.default_rng(cfg.split_seed)
    pattern = np.array(["train"] * cfg.split_pattern[0] + ["val"] * cfg.split_pattern[1] + ["test"] * cfg.split_pattern[2])
    rng.shuffle(pattern)
    order = []
    for s in sorted(g["stratum"].unique()):
        names = np.array(sorted(g.index[g["stratum"] == s]))
        rng.shuffle(names)
        order.extend(names.tolist())
    g.loc[order, "split"] = pattern[np.arange(len(order)) % block]
    df["split"] = df.group.map(g["split"])
    df["stratum"] = df.group.map(g["stratum"])
    assert (df.groupby("subject_id")["split"].nunique() == 1).all(), "a subject appears in two partitions"
    assert (df.groupby("group")["split"].nunique() == 1).all(), "a group appears in two partitions"
    cols = ["case_id", "subject_id", "timepoint", "group", "split", "stratum", "et_present", "site"]
    return df[cols].sort_values("case_id").reset_index(drop=True)


def qc_overlay(case_ids, out_png, notes=None):
    case_ids = list(case_ids)[:24]
    if not case_ids:
        return
    fig, axes = plt.subplots(len(case_ids), len(CFG.modalities), figsize=(2.2 * len(CFG.modalities), 2.2 * len(case_ids)),
                             squeeze=False)
    for r, cid in enumerate(case_ids):
        img = np.load(DIRS["cache"] / cid / "img.npy", mmap_mode="r")
        seg = np.load(DIRS["cache"] / cid / "seg.npy", mmap_mode="r")
        area = np.asarray(seg).reshape(seg.shape[0], -1).astype(bool).sum(1)
        z = int(area.argmax()) if area.any() else img.shape[0] // 2
        for c, m in enumerate(CFG.modalities):
            ax = axes[r, c]
            ax.imshow(np.asarray(img[z, c], np.float32).T, cmap="gray", origin="lower")
            if area.any():
                ax.contour(np.asarray(seg[z]).T, levels=[0.5, 1.5, 2.5], colors=["yellow", "red", "cyan"], linewidths=0.6)
            ax.set_axis_off()
            title = f"{cid[-9:]} {m}" if c == 0 else m
            if c == 0 and notes and notes.get(cid):
                title += "\n" + notes[cid][:40]
            ax.set_title(title, fontsize=7)
    fig.tight_layout()
    fig.savefig(out_png, dpi=110)
    plt.close(fig)

In [ ]:
AUDIT_CSV, SPLIT_CSV, SITES_CSV = DIRS["audit"] / "audit.csv", DIRS["audit"] / "split.csv", DIRS["audit"] / "sites.csv"
if not AUDIT_CSV.exists():
    case_dirs, mapping_path = locate_training_data(DATA_ROOT, CFG)
    audit_raw, fps = run_audit(case_dirs, CFG, DIRS["cache"])
    audit, near_pairs, modal = cohort_checks(audit_raw, fps, CFG)
    sites = read_site_table(mapping_path, CFG)
    if sites is not None:
        sites.to_csv(SITES_CSV, index=False)
    audit.to_csv(AUDIT_CSV, index=False)
    near_pairs.to_csv(DIRS["audit"] / "similar_pairs.csv", index=False)
    audit.loc[audit.status == HARD, ["case_id", "reasons"]].to_csv(DIRS["audit"] / "exclusions.csv", index=False)
    audit.loc[audit.review != "", ["case_id", "review"]].to_csv(DIRS["audit"] / "review.csv", index=False)
    for c in audit.loc[audit.status == HARD, "case_id"]:
        shutil.rmtree(DIRS["cache"] / c, ignore_errors=True)
    flagged = audit.loc[(audit.review != "") & (audit.status != HARD)]
    qc_overlay(flagged.case_id, DIRS["qc"] / "review_cases.png", dict(zip(flagged.case_id, flagged.review)))
    ok_ids = audit.loc[audit.status != HARD, "case_id"]
    qc_overlay(ok_ids.sample(n=min(8, len(ok_ids)), random_state=CFG.split_seed), DIRS["qc"] / "random_cases.png")
    print("cohort modal header values:", modal)


def load_audit():
    a = pd.read_csv(AUDIT_CSV, dtype={"subject_id": str})
    for col in ("reasons", "review", "group"):
        a[col] = a[col].fillna("").astype(str)
    return a


audit = load_audit()
print("status counts:", audit.status.value_counts().to_dict())
print(audit.loc[audit.status == HARD, ["case_id", "reasons"]].to_string(index=False))
print("\ncases flagged for visual review (see qc/review_cases.png):")
print(audit.loc[audit.review != "", ["case_id", "review"]].to_string(index=False))
neg = {m: int((pd.to_numeric(audit[f"negative_frac_{m}"], errors="coerce") > 0).sum()) for m in CFG.modalities}
print("\ncases with negative intensities per modality:", neg)

In [ ]:
if SMOKE_TEST:
    got = set(audit.loc[audit.status == HARD, "case_id"])
    assert got == set(PHANTOM["exclude"]), f"exclusions {sorted(got)} differ from planted {sorted(PHANTOM['exclude'])}"
    flagged = set(audit.loc[audit.review != "", "case_id"])
    missed = set(PHANTOM["review"]) - flagged
    assert not missed, f"planted modality swaps not flagged: {missed}"
    extra = sorted(flagged - set(PHANTOM["review"]))
    print("PASS: every planted defect was caught; extra review flags (false alarms):", extra if extra else "none")

### Visual review decisions, then the split

Open `qc/review_cases.png`, decide for every flagged case, and record each exclusion with its reason in `MANUAL_EXCLUDE`. Flags you keep are simply left out of the dictionary. The split is built once and hashed; if you add an exclusion after the split exists, the cell stops and asks you to delete `split.csv`, which is only acceptable before any training has started. In smoke test mode the planted modality swaps play the role of confirmed findings.

In [ ]:
MANUAL_EXCLUDE = {}   # {"case_id": "reason seen on the overlay"}
if SMOKE_TEST:
    MANUAL_EXCLUDE = {c: f"confirmed on overlay: {why}" for c, why in PHANTOM["review"].items()}


def apply_manual_exclusions(a, manual):
    a = a.copy()
    unknown = set(manual) - set(a.case_id)
    assert not unknown, f"unknown case IDs in MANUAL_EXCLUDE: {unknown}"
    for c, why in manual.items():
        a.loc[a.case_id == c, ["status", "reasons"]] = [HARD, f"manual review: {why}"]
    return a


if SPLIT_CSV.exists():
    split = pd.read_csv(SPLIT_CSV, dtype={"subject_id": str})
    stale = sorted(set(split.case_id) & set(MANUAL_EXCLUDE))
    if stale:
        raise RuntimeError(f"{stale} are excluded but still in split.csv; delete split.csv and rerun (before training only)")
else:
    sites = pd.read_csv(SITES_CSV, dtype=str) if SITES_CSV.exists() else None
    split = make_split(apply_manual_exclusions(audit, MANUAL_EXCLUDE), sites, CFG)
    split.to_csv(SPLIT_CSV, index=False)
    pd.DataFrame({"case_id": list(MANUAL_EXCLUDE), "reason": list(MANUAL_EXCLUDE.values())}).to_csv(
        DIRS["audit"] / "manual_exclusions.csv", index=False)
SPLIT_SHA256 = sha256_file(SPLIT_CSV)
print(f"N clean = {len(split)} cases, split sha256 {SPLIT_SHA256}")
print(split.groupby("split").agg(cases=("case_id", "size"), groups=("group", "nunique"),
                                 et_present=("et_present", "mean"), sites=("site", "nunique")))
if SMOKE_TEST:
    s = split.set_index("case_id")["split"]
    for a_, b_ in PHANTOM["same_partition"]:
        assert s[a_] == s[b_], "two timepoints of one subject landed in different partitions"
    print("PASS: both timepoints of subject 00003 are in the same partition")

## Stage B. Known answer tests for metrics, inputs and losses

Nothing downstream should run until this section passes. The tests cover analytic Dice and HD95 values, the BraTS empty mask conventions, anisotropic spacing, crop invariance, agreement with MONAI on random shapes, the 2.5D channel layout and the dropped modality bookkeeping, identical inputs on the training path (CPU) and the inference path (device), the export geometry (cache back to the original NIfTI grid), the deterministic sampling plan, the augmentation identity, and the consistency loss.

In [ ]:
CROSS3 = ndimage.generate_binary_structure(3, 1)


def regions_from_labels(lab):
    """Labels 0..3 (0 background, 1 NCR, 2 ED, 3 ET) to stacked boolean WT, TC, ET masks."""
    lab = np.asarray(lab)
    return np.stack([lab > 0, (lab == 1) | (lab == 3), lab == 3])


def region_prob(p, r):
    """Region probability from the four class softmax: WT = p1 + p2 + p3, TC = p1 + p3, ET = p3."""
    return p[1] + p[2] + p[3] if r == 0 else (p[1] + p[3] if r == 1 else p[3])


def bbox_slices(mask, margin):
    sl = []
    for ax in range(mask.ndim):
        idx = np.where(mask.any(axis=tuple(i for i in range(mask.ndim) if i != ax)))[0]
        sl.append(slice(max(0, idx[0] - margin), idx[-1] + 1 + margin))
    return tuple(sl)


def surface(mask):
    return mask & ~ndimage.binary_erosion(mask, CROSS3)


def hd95_np(pred, gt, spacing=(1.0, 1.0, 1.0), penalty=373.13, crop=True):
    """Symmetric HD95: the larger of the two directed 95th percentile surface distances (MONAI convention)."""
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    if not pred.any() and not gt.any():
        return 0.0
    if not pred.any() or not gt.any():
        return float(penalty)
    if crop:
        sl = bbox_slices(pred | gt, 2)
        pred, gt = pred[sl], gt[sl]
    sp, sg = surface(pred), surface(gt)
    d_pg = ndimage.distance_transform_edt(~sg, sampling=spacing)[sp]
    d_gp = ndimage.distance_transform_edt(~sp, sampling=spacing)[sg]
    return float(max(np.percentile(d_pg, 95), np.percentile(d_gp, 95)))


def to_full_volume(meta, lab):
    """Cached crop (Z, X, Y) back to the original (X, Y, Z) grid."""
    full = np.zeros(meta["shape"], np.uint8)
    (x0, x1), (y0, y1), (z0, z1) = meta["bbox"]
    full[x0:x1, y0:y1, z0:z1] = np.asarray(lab).transpose(1, 2, 0)
    return full

In [ ]:
def canvas_from_meta(meta):
    up = lambda n: int(math.ceil(n / 16) * 16)
    return (up(max(m["bbox"][0][1] - m["bbox"][0][0] for m in meta.values())),
            up(max(m["bbox"][1][1] - m["bbox"][1][0] for m in meta.values())))


def pad_to_canvas(a, canvas):
    X, Y = a.shape[-2:]
    H, W = canvas
    if X > H or Y > W:
        raise ValueError(f"slice {X}x{Y} larger than canvas {H}x{W}")
    px, py = (H - X) // 2, (W - Y) // 2
    out = np.zeros(a.shape[:-2] + (H, W), a.dtype)
    out[..., px:px + X, py:py + Y] = a
    return out


def unpad(t, shape, canvas):
    (X, Y), (H, W) = shape, canvas
    px, py = (H - X) // 2, (W - Y) // 2
    return t[..., px:px + X, py:py + Y]


def build_25d(vol, z):
    """vol: (Z, 4, X, Y). Returns (12, X, Y), modality major: for each modality the slice below, slice z itself
    and the slice above. Indices are clamped, so the first and last cached slices repeat (proposal 2.3)."""
    nz = vol.shape[0]
    zs = [min(max(z + d, 0), nz - 1) for d in (-1, 0, 1)]
    x = np.asarray(vol[zs], dtype=np.float32)
    return x.transpose(1, 0, 2, 3).reshape(-1, *x.shape[2:])


def stack_25d_torch(vol_t, zs):
    """Same layout as build_25d, built on the device from a whole padded volume (Z, 4, H, W) for slices zs."""
    nz = vol_t.shape[0]
    nb = torch.stack([(zs - 1).clamp(0, nz - 1), zs, (zs + 1).clamp(0, nz - 1)], 1)    # (b, 3)
    x = vol_t[nb]                                                                        # (b, 3, 4, H, W)
    return x.permute(0, 2, 1, 3, 4).reshape(len(zs), -1, *vol_t.shape[-2:]).float()


def make_inputs(x, drop=None):
    """x: (B, 12, H, W). drop: None (complete) or (B,) modality indices to remove.
    Returns (B, 16, H, W): 12 image channels with the dropped modality zeroed, then 4 availability maps."""
    B, _, H, W = x.shape
    n_mod = x.shape[1] // 3
    avail = torch.ones(B, n_mod, device=x.device, dtype=x.dtype)
    if drop is not None:
        avail[torch.arange(B, device=x.device), drop] = 0
    x = x * avail.repeat_interleave(3, dim=1)[:, :, None, None]
    return torch.cat([x, avail[:, :, None, None].expand(B, n_mod, H, W)], 1)


class SliceDataset(Dataset):
    def __init__(self, cache_dir, items, canvas):
        self.cache_dir, self.items, self.canvas = Path(cache_dir), items, canvas

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        cid, z = self.items[i]
        vol = np.load(self.cache_dir / cid / "img.npy", mmap_mode="r")
        seg = np.load(self.cache_dir / cid / "seg.npy", mmap_mode="r")
        x = pad_to_canvas(build_25d(vol, z), self.canvas)
        y = pad_to_canvas(np.asarray(seg[z], dtype=np.int64), self.canvas)
        return torch.from_numpy(x), torch.from_numpy(y), i


def epoch_plan(train_ids, meta, cfg, seed, epoch):
    """Everything random about one epoch, drawn from (seed, epoch) only. Arms that share a seed therefore see
    identical slices, augmentations and dropped modalities (common random numbers)."""
    rng = np.random.default_rng([seed, epoch, 20260925])
    n_t = int(round(cfg.slices_per_patient * cfg.tumor_fraction))
    items = []
    for cid in train_ids:
        ts, bs = meta[cid]["tumor_slices"], meta[cid]["brain_slices"]
        zs = np.concatenate([rng.choice(ts if ts else bs, n_t), rng.choice(bs, cfg.slices_per_patient - n_t)])
        items += [(cid, int(z)) for z in zs]
    items = [items[i] for i in rng.permutation(len(items))]
    n = len(items)
    return {"items": items,
            "drop": rng.integers(0, len(cfg.modalities), n),
            "flip": rng.random((n, 2)) < cfg.p_flip,
            "angle": np.where(rng.random(n) < cfg.p_rotate, rng.uniform(-1, 1, n) * np.deg2rad(cfg.max_rotate_deg), 0.0),
            "scale": np.where(rng.random(n) < cfg.p_scale, rng.uniform(*cfg.scale_range, n), 1.0)}


def augment(x, y, flip, angle, scale):
    """One random in plane affine per sample, applied to the 12 image channels (bilinear) and the label (nearest).
    The rotation is corrected for the canvas aspect ratio, so it stays rigid in pixel space."""
    B, _, H, W = x.shape
    c, s = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0], theta[:, 0, 1] = c, -s * H / W
    theta[:, 1, 0], theta[:, 1, 1] = s * W / H, c
    theta[:, :, :2] = theta[:, :, :2] / scale[:, None, None]
    theta[:, :, 0] = theta[:, :, 0] * torch.where(flip[:, 0], -1.0, 1.0)[:, None]
    theta[:, :, 1] = theta[:, :, 1] * torch.where(flip[:, 1], -1.0, 1.0)[:, None]
    grid = F.affine_grid(theta, (B, 1, H, W), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    y = F.grid_sample(y[:, None].float(), grid, mode="nearest", padding_mode="zeros", align_corners=False)
    return x, y[:, 0].round().long()


def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm2d(cout, affine=True), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm2d(cout, affine=True), nn.ReLU(inplace=True))


class UNet25D(nn.Module):
    def __init__(self, in_ch=16, n_classes=4, features=(32, 64, 128, 256), bottleneck=512):
        super().__init__()
        self.down, c = nn.ModuleList(), in_ch
        for f in features:
            self.down.append(conv_block(c, f))
            c = f
        self.pool = nn.MaxPool2d(2)
        self.mid = conv_block(c, bottleneck)
        self.up, self.dec, c = nn.ModuleList(), nn.ModuleList(), bottleneck
        for f in reversed(features):
            self.up.append(nn.ConvTranspose2d(c, f, 2, stride=2))
            self.dec.append(conv_block(2 * f, f))
            c = f
        self.head = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.mid(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)


def build_model(cfg=None):
    cfg = cfg or CFG
    n = len(cfg.modalities)
    return UNet25D(in_ch=4 * n, n_classes=4, features=cfg.features, bottleneck=cfg.bottleneck)


def seg_loss(logits, y, eps=1e-5):
    """Equation (1): cross entropy plus one minus the mean soft Dice over NCR, ED, ET, pooled over the batch."""
    logits = logits.float()
    p = logits.softmax(1)
    onehot = F.one_hot(y, logits.shape[1]).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    inter = (p * onehot).sum(dims)[1:]
    denom = p.sum(dims)[1:] + onehot.sum(dims)[1:]
    return F.cross_entropy(logits, y) + 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def consistency_loss(logits_full, logits_miss, mask):
    """Equation (2): mean over brain voxels of KL(sg(p_full) || p_miss)."""
    lp_full = F.log_softmax(logits_full.float(), 1).detach()
    lp_miss = F.log_softmax(logits_miss.float(), 1)
    kl = (lp_full.exp() * (lp_full - lp_miss)).sum(1)
    m = mask.float()
    return (kl * m).sum() / m.sum().clamp_min(1.0)

In [ ]:
def run_known_answer_tests():
    report = {}
    z = np.zeros((32, 32, 32), bool)
    a, b = z.copy(), z.copy()
    a[8:18, 8:18, 8:18] = True
    b[13:23, 8:18, 8:18] = True
    assert abs(dice_np(a, b) - 0.5) < 1e-12 and dice_np(z, z) == 1.0 and dice_np(a, z) == 0.0
    p, q = z.copy(), z.copy()
    p[10, 10, 10] = True
    q[10, 10, 15] = True
    assert abs(hd95_np(p, q, (1.0, 1.0, 2.0)) - 10.0) < 1e-9, "anisotropic spacing"
    assert hd95_np(a, a) == 0.0 and hd95_np(z, z) == 0.0 and hd95_np(a, z) == 373.13 and hd95_np(z, a) == 373.13
    assert abs(hd95_np(a, b) - hd95_np(b, a)) < 1e-12 and 0 < hd95_np(a, b) <= 5.0 + 1e-9
    rng = np.random.default_rng(0)
    blobs = []
    for _ in range(12):
        f1 = ndimage.gaussian_filter(rng.random((40, 44, 36)), 3)
        f2 = ndimage.gaussian_filter(rng.random((40, 44, 36)), 3)
        blobs.append((f1 > np.quantile(f1, 0.85), f2 > np.quantile(f2, 0.85)))
    for m1, m2 in blobs:
        assert abs(hd95_np(m1, m2, (1.0, 1.2, 1.5)) - hd95_np(m1, m2, (1.0, 1.2, 1.5), crop=False)) < 1e-9, "crop changed HD95"
    r = regions_from_labels(np.array([0, 1, 2, 3]))
    assert r[0].tolist() == [False, True, True, True] and r[1].tolist() == [False, True, False, True] and r[2].tolist() == [False, False, False, True]
    report["analytic"] = "pass"

    try:
        from monai.metrics import compute_hausdorff_distance, compute_dice
        diffs_d, diffs_h = [], []
        for spacing in ((1.0, 1.0, 1.0), (1.0, 1.2, 1.5)):
            for m1, m2 in blobs:
                yp = torch.from_numpy(np.stack([~m1, m1])[None].astype(np.float32))
                yt = torch.from_numpy(np.stack([~m2, m2])[None].astype(np.float32))
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    h = compute_hausdorff_distance(yp, yt, include_background=False, percentile=95, spacing=list(spacing))
                    d = compute_dice(yp, yt, include_background=False)
                diffs_h.append(abs(float(h) - hd95_np(m1, m2, spacing)))
                diffs_d.append(abs(float(d) - dice_np(m1, m2)))
        report["monai_max_abs_diff"] = {"dice": max(diffs_d), "hd95": max(diffs_h)}
        assert max(diffs_d) < 1e-5 and max(diffs_h) < 1e-4, f"disagreement with MONAI: {report['monai_max_abs_diff']}"
    except ImportError:
        report["monai_max_abs_diff"] = "MONAI not installed: install it and rerun before trusting any metric"
        warnings.warn(report["monai_max_abs_diff"])

    vol = np.zeros((5, 4, 6, 7), np.float16)
    for zz in range(5):
        for m in range(4):
            vol[zz, m] = 10 * m + zz + 1
    x0 = build_25d(vol, 0)
    x4 = build_25d(vol, 4)
    for m in range(4):
        assert x0[3 * m:3 * m + 3, 0, 0].tolist() == [10 * m + 1, 10 * m + 1, 10 * m + 2]
        assert x4[3 * m:3 * m + 3, 0, 0].tolist() == [10 * m + 4, 10 * m + 5, 10 * m + 5]
    xt = torch.from_numpy(build_25d(vol, 2))[None]
    for m in range(4):
        inp = make_inputs(xt, torch.tensor([m]))
        zero = [ch for ch in range(12) if float(inp[0, ch].abs().sum()) == 0.0]
        assert zero == [3 * m, 3 * m + 1, 3 * m + 2], f"dropping modality {m} zeroed channels {zero}"
        assert float(inp[0, 12 + m].abs().sum()) == 0.0 and all(float(inp[0, 12 + k].min()) == 1.0 for k in range(4) if k != m)
    full = make_inputs(xt)
    assert torch.equal(full[:, :12], xt) and float(full[:, 12:].min()) == 1.0
    canvas = (16, 16)
    vt = torch.from_numpy(pad_to_canvas(vol, canvas))
    for zz in range(5):
        a_ = torch.from_numpy(pad_to_canvas(build_25d(vol, zz), canvas))
        b_ = stack_25d_torch(vt, torch.tensor([zz]))[0]
        assert torch.equal(a_, b_), "training and inference paths build different inputs"
    report["inputs"] = "pass"

    img = torch.from_numpy(rng.standard_normal((2, 12, 32, 48)).astype(np.float32))
    lab = torch.from_numpy(rng.integers(0, 4, (2, 32, 48)))
    xi, yi = augment(img, lab, torch.zeros(2, 2, dtype=torch.bool), torch.zeros(2), torch.ones(2))
    assert torch.allclose(xi, img, atol=1e-5) and torch.equal(yi, lab), "identity augmentation changed the data"
    lg = torch.randn(3, 4, 8, 8)
    assert abs(float(consistency_loss(lg, lg, torch.ones(3, 8, 8)))) < 1e-6
    assert float(consistency_loss(lg, torch.randn(3, 4, 8, 8), torch.ones(3, 8, 8))) > 0
    report["augment_and_losses"] = "pass"

    cases = split.case_id.tolist()
    metas = {c: json.loads((DIRS["cache"] / c / "meta.json").read_text()) for c in cases[:3]}
    p1 = epoch_plan(cases[:3], metas, CFG, 11, 3)
    p2 = epoch_plan(cases[:3], metas, CFG, 11, 3)
    p3 = epoch_plan(cases[:3], metas, CFG, 22, 3)
    assert p1["items"] == p2["items"] and np.array_equal(p1["drop"], p2["drop"]) and np.array_equal(p1["angle"], p2["angle"])
    assert p1["items"] != p3["items"]
    tumor_share = np.mean([z in metas[c]["tumor_slices"] for c, z in p1["items"]])
    assert tumor_share >= 0.45, tumor_share
    report["sampling_plan"] = f"pass (tumor slice share {tumor_share:.2f})"

    c0 = cases[0]
    meta0 = metas[c0]
    case_dir = next(Path(DATA_ROOT).rglob(c0))
    orig = np.asarray(nib.load(str(next(case_dir.glob(f"{c0}-seg.nii*")))).dataobj).astype(np.uint8)
    assert np.array_equal(to_full_volume(meta0, np.load(DIRS["cache"] / c0 / "seg.npy")), orig), "cache to NIfTI geometry"
    report["export_geometry"] = "pass"
    return report


print(json.dumps(run_known_answer_tests(), indent=1))

## Training and evaluation machinery

`train_run` is resumable (it reloads `last.pt` after an interruption) and returns immediately if the run has already finished. `evaluate` computes 3D metrics per case, condition and region in the cropped space, which equals the full volume result because the crop contains every brain and label voxel.

In [ ]:
@dataclass
class CohortData:
    split: pd.DataFrame
    meta: dict
    canvas: tuple
    train_ids: list
    val_ids: list
    test_ids: list


def load_cohort():
    sp = pd.read_csv(SPLIT_CSV, dtype={"subject_id": str})
    meta = {c: json.loads((DIRS["cache"] / c / "meta.json").read_text()) for c in sp.case_id}
    ids = {s: sp.loc[sp.split == s, "case_id"].tolist() for s in ("train", "val", "test")}
    return CohortData(sp, meta, canvas_from_meta(meta), ids["train"], ids["val"], ids["test"])


def run_name(arm, seed, lam=0.0):
    return f"{arm}_seed{seed}" + (f"_lam{lam:g}" if arm == "M2" else "")


def load_best(name):
    model = build_model().to(DEVICE)
    ck = torch.load(DIRS["runs"] / name / "best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    return model.eval()


def volume_to_device(vol, canvas):
    """Whole cached crop (Z, 4, X, Y) padded to the canvas and moved to the device once per case."""
    return torch.from_numpy(pad_to_canvas(np.asarray(vol), canvas)).to(DEVICE)


@torch.no_grad()
def predict_volume(model, vol_t, shape, canvas, drop=None, want_probs=False):
    """vol_t: padded volume on the device. Returns softmax probabilities (4, Z, X, Y) or labels (Z, X, Y)."""
    nz, (X, Y) = vol_t.shape[0], shape
    out = np.empty((4, nz, X, Y), np.float32) if want_probs else np.empty((nz, X, Y), np.uint8)
    model.eval()
    for z0 in range(0, nz, CFG.eval_batch):
        zs = torch.arange(z0, min(nz, z0 + CFG.eval_batch), device=DEVICE)
        d = None if drop is None else torch.full((len(zs),), drop, device=DEVICE, dtype=torch.long)
        with autocast():
            logits = unpad(model(make_inputs(stack_25d_torch(vol_t, zs), d)).float(), (X, Y), canvas)
        if want_probs:
            out[:, z0:z0 + len(zs)] = logits.softmax(1).permute(1, 0, 2, 3).cpu().numpy()
        else:
            out[z0:z0 + len(zs)] = logits.argmax(1).to(torch.uint8).cpu().numpy()
    return out


def evaluate(model, case_ids, data, conditions=CONDITIONS, full=False, export_dir=None, prob_cases=(), calib=None, tag="run"):
    """Dice per case, condition and region. full=True adds HD95, volumes, agreement with and KL to the complete
    input prediction (manipulation check), far false positives, near tumor Brier score and reliability counts."""
    rows = []
    for cid in case_ids:
        meta = data.meta[cid]
        vol = np.load(DIRS["cache"] / cid / "img.npy")
        vol_t, shape = volume_to_device(vol, data.canvas), vol.shape[-2:]
        gt = np.load(DIRS["cache"] / cid / "seg.npy")
        sx, sy, sz = meta["spacing"]
        spacing, vox_ml = (sz, sx, sy), sx * sy * sz / 1000.0
        penalty = round(float(np.linalg.norm(np.array(meta["shape"]) * np.array(meta["spacing"]))), 2)
        G = regions_from_labels(gt)
        if full:
            brain = np.abs(vol.astype(np.float32)).sum(1) > 0
            far = ndimage.distance_transform_edt(~G[0], sampling=spacing) > CFG.far_fp_mm if G[0].any() else np.ones_like(G[0])
        ref_p = ref_P = None
        for cond in conditions:
            drop = None if cond == "complete" else CFG.modalities.index(cond[3:])
            if full:
                p = predict_volume(model, vol_t, shape, data.canvas, drop, want_probs=True)
                lab = p.argmax(0).astype(np.uint8)
            else:
                lab = predict_volume(model, vol_t, shape, data.canvas, drop)
            P = regions_from_labels(lab)
            kl = np.nan
            if full and ref_p is not None:
                pf, pm = ref_p[:, brain], p[:, brain]
                kl = float((pf * (np.log(pf + 1e-7) - np.log(pm + 1e-7))).sum(0).mean())
            for r, reg in enumerate(REGIONS):
                row = {"case_id": cid, "condition": cond, "region": reg, "dice": dice_np(P[r], G[r])}
                if full:
                    u = P[r] | G[r]
                    if u.any():
                        sl = bbox_slices(u, 5)
                        roi = ndimage.binary_dilation(u[sl], iterations=3)
                        q, y = region_prob(p[(slice(None),) + sl], r)[roi], G[r][sl][roi]
                    else:
                        q = y = np.zeros(0)
                    row.update(hd95=hd95_np(P[r], G[r], spacing, penalty), pred_ml=float(P[r].sum() * vox_ml),
                               gt_ml=float(G[r].sum() * vox_ml), far_fp_ml=float(np.count_nonzero(P[r] & far) * vox_ml),
                               brier_roi=float(np.mean((q - y) ** 2)) if q.size else np.nan, kl_to_complete=kl,
                               agree_dice=dice_np(P[r], ref_P[r]) if ref_P is not None else np.nan)
                    if calib is not None and q.size:
                        acc = calib.setdefault((cond, reg), np.zeros((3, 10)))
                        b = np.minimum((q * 10).astype(int), 9)
                        acc[0] += np.bincount(b, minlength=10)
                        acc[1] += np.bincount(b, weights=q, minlength=10)
                        acc[2] += np.bincount(b, weights=y.astype(float), minlength=10)
                rows.append(row)
            if cond == "complete" and full:
                ref_p, ref_P = p, P
            if export_dir is not None:
                d = Path(export_dir) / cond
                d.mkdir(parents=True, exist_ok=True)
                nib.save(nib.Nifti1Image(to_full_volume(meta, lab), np.array(meta["affine"])), str(d / f"{cid}.nii.gz"))
            if full and cid in prob_cases:
                d = DIRS["eval"] / "probs"
                d.mkdir(parents=True, exist_ok=True)
                q8 = np.stack([np.round(255 * region_prob(p, r)).astype(np.uint8) for r in range(3)])
                np.savez_compressed(d / f"{tag}_{cid}_{cond}.npz", q=q8)
    return pd.DataFrame(rows)


def train_run(arm, seed, lam, data, cfg=None):
    cfg = cfg or CFG
    name = run_name(arm, seed, lam)
    out = DIRS["runs"] / name
    out.mkdir(parents=True, exist_ok=True)
    if (out / "summary.json").exists():
        return json.loads((out / "summary.json").read_text())
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    model = build_model(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    st = {"epoch": 0, "best": -1.0, "best_epoch": 0, "bad": 0, "history": [], "seconds": 0.0, "steps": 0, "peak_gb": 0.0}
    if (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        st = ck["state"]
        print(f"{name}: resuming after epoch {st['epoch']}")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    while st["epoch"] < cfg.max_epochs and st["bad"] < cfg.patience_checks:
        epoch = st["epoch"]
        plan = epoch_plan(data.train_ids, data.meta, cfg, seed, epoch)
        loader = DataLoader(SliceDataset(DIRS["cache"], plan["items"], data.canvas), batch_size=cfg.batch_size,
                            shuffle=False, drop_last=True, num_workers=cfg.num_workers, pin_memory=DEVICE.type == "cuda")
        model.train()
        t0, loss_sum, n_steps = time.time(), torch.zeros((), device=DEVICE), 0
        for x, y, idx in loader:
            x, y, idx = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), idx.numpy()
            x, y = augment(x, y, torch.as_tensor(plan["flip"][idx], device=DEVICE),
                           torch.as_tensor(plan["angle"][idx], device=DEVICE, dtype=torch.float32),
                           torch.as_tensor(plan["scale"][idx], device=DEVICE, dtype=torch.float32))
            drop = torch.as_tensor(plan["drop"][idx], device=DEVICE)
            with autocast():
                if arm == "M0":
                    loss = seg_loss(model(make_inputs(x)), y)
                else:
                    logits = model(torch.cat([make_inputs(x), make_inputs(x, drop)]))  # instance norm: same as two passes
                    lf, lm = logits[: len(x)], logits[len(x):]
                    loss = 0.5 * (seg_loss(lf, y) + seg_loss(lm, y))
                    if arm == "M2":
                        loss = loss + lam * consistency_loss(lf, lm, x.abs().sum(1) > 0)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.detach()
            n_steps += 1
            if n_steps % cfg.log_every == 0:
                rate = (time.time() - t0) / n_steps
                print(f"  {name} epoch {epoch + 1} step {n_steps}/{len(loader)}: loss {float(loss_sum) / n_steps:.4f}, "
                      f"{rate:.3f} s/step, epoch eta {rate * (len(loader) - n_steps) / 60:.1f} min")
        seconds = time.time() - t0
        st["seconds"] += seconds
        st["steps"] += n_steps
        st["epoch"] = epoch + 1
        rec = {"epoch": epoch + 1, "train_loss": float(loss_sum) / max(1, n_steps), "epoch_seconds": round(seconds, 1)}
        if (epoch + 1) % cfg.val_every == 0 or epoch + 1 == cfg.max_epochs:
            score = float(evaluate(model, data.val_ids, data)["dice"].mean())
            rec["val_score"] = score
            if score > st["best"]:
                st.update(best=score, best_epoch=epoch + 1, bad=0)
                torch.save({"model": model.state_dict(), "epoch": epoch + 1, "val_score": score}, out / "best.pt")
            else:
                st["bad"] += 1
        st["history"].append(rec)
        if DEVICE.type == "cuda":
            st["peak_gb"] = max(st["peak_gb"], torch.cuda.max_memory_allocated() / 2 ** 30)
        if epoch == 0 or "val_score" in rec:
            print(f"{name}: " + ", ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "state": st}, out / "last.pt")
    summary = {"name": name, "arm": arm, "seed": seed, "lambda": lam, "best_val_score": st["best"], "best_epoch": st["best_epoch"],
               "epochs_run": st["epoch"], "steps": st["steps"], "train_hours": st["seconds"] / 3600,
               "sec_per_step": st["seconds"] / max(1, st["steps"]), "peak_gpu_gb": st["peak_gb"],
               "parameters": sum(p.numel() for p in model.parameters()), "history": st["history"],
               "checkpoint_sha256": sha256_file(out / "best.pt"), "finished": stamp()}
    (out / "summary.json").write_text(json.dumps(summary, indent=1))
    (out / "last.pt").unlink(missing_ok=True)
    return summary


DATA = load_cohort()
print(f"canvas {DATA.canvas}; cases train {len(DATA.train_ids)}, val {len(DATA.val_ids)}, test {len(DATA.test_ids)}")
print(f"parameters: {sum(p.numel() for p in build_model().parameters()):,}")

## Stage C. $\lambda$ selection on tuning seed 0

Validation data only. M1 at the tuning seed is trained as a reference; if no $\lambda$ beats it on validation, that is worth reporting, but the pre specified rule still picks the best of the candidate values.

In [ ]:
LAMBDA_FILE = DIRS["runs"] / "lambda_selection.json"
if not LAMBDA_FILE.exists():
    ref = train_run("M1", CFG.tuning_seed, 0.0, DATA)
    cands = [train_run("M2", CFG.tuning_seed, lam, DATA) for lam in CFG.lambdas]
    best = max(cands, key=lambda s: s["best_val_score"])
    LAMBDA_FILE.write_text(json.dumps({"lambda": best["lambda"], "tuning_seed": CFG.tuning_seed,
                                       "validation_scores": {f"{s['lambda']:g}": s["best_val_score"] for s in cands},
                                       "reference_M1": ref["best_val_score"], "selected": stamp()}, indent=1))
SEL = json.loads(LAMBDA_FILE.read_text())
LAMBDA = SEL["lambda"]
print(json.dumps(SEL, indent=1))

## Stage D. Precision analysis before the test set is touched

The per patient paired difference (M2 minus M1, averaged over the four missing conditions and three regions) on validation data gives the standard deviation that drives the width of the final confidence interval. With one tuning seed this is conservative, because the final estimate averages three seeds. If the minimum detectable effect is larger than any effect you would care about, say so in the report before seeing the test result.

In [ ]:
PRECISION_FILE = DIRS["analysis"] / "precision.json"
if not PRECISION_FILE.exists():
    v = {arm: evaluate(load_best(run_name(arm, CFG.tuning_seed, lam)), DATA.val_ids, DATA, conditions=MISSING)
         for arm, lam in (("M1", 0.0), ("M2", LAMBDA))}
    m = v["M2"].merge(v["M1"], on=["case_id", "condition", "region"], suffixes=("_m2", "_m1"))
    m["group"] = m.case_id.map(DATA.split.set_index("case_id")["group"])
    d = (m.dice_m2 - m.dice_m1).groupby(m.group).mean()
    n_test = DATA.split.loc[DATA.split.split == "test", "group"].nunique()
    sd = float(d.std(ddof=1)) if len(d) > 1 else float("nan")
    se = sd / math.sqrt(n_test)
    PRECISION_FILE.write_text(json.dumps({
        "n_validation_patients": int(len(d)), "validation_mean_difference": float(d.mean()), "sd_patient_difference": sd,
        "n_test_patients": int(n_test), "projected_ci_half_width": 1.96 * se,
        "minimum_detectable_effect_80pct_power": (stats.norm.ppf(0.975) + stats.norm.ppf(0.8)) * se,
        "noninferiority_margin": CFG.ni_margin, "computed": stamp()}, indent=1))
print(PRECISION_FILE.read_text())

## Stage E. Final runs

Nine runs: M0, M1 and M2 for each of seeds 11, 22 and 33, with the selected $\lambda$. Runs can be split across sessions; each call returns immediately if that run has finished and resumes it if it was interrupted. The cost table at the end is the meaningful compute comparison.

In [ ]:
FINAL_RUNS = [(arm, seed, LAMBDA if arm == "M2" else 0.0) for seed in CFG.seeds for arm in ARMS]
SUMMARIES = [train_run(a, s, l, DATA) for a, s, l in FINAL_RUNS]
cost = pd.DataFrame(SUMMARIES)[["name", "arm", "seed", "lambda", "best_val_score", "best_epoch", "epochs_run",
                                "steps", "train_hours", "sec_per_step", "peak_gpu_gb", "parameters"]]
cost.to_csv(DIRS["analysis"] / "training_cost.csv", index=False)
print(cost.to_string(index=False))
print(cost.groupby("arm")[["train_hours", "sec_per_step", "best_val_score"]].mean())

fig, ax = plt.subplots(figsize=(6, 3.5))
for s in SUMMARIES:
    h = [r for r in s["history"] if "val_score" in r]
    ax.plot([r["epoch"] for r in h], [r["val_score"] for r in h], marker="o", ms=3, label=s["name"])
ax.set_xlabel("epoch")
ax.set_ylabel("validation score")
ax.legend(fontsize=6, ncol=3)
fig.tight_layout()
fig.savefig(DIRS["analysis"] / "validation_curves.png", dpi=120)
plt.show()

## Stage F. Analysis code frozen, lockbox written, one test evaluation

The analysis functions are defined **before** the test set is evaluated. The lockbox records the split hash, $\lambda$, every checkpoint hash and the hash of the analysis code. Test evaluation refuses to run if a checkpoint changed after the lockbox was written, and the lockbox refuses to be rewritten with different analysis code. If you genuinely need to change the analysis afterwards, report it as a deviation.

In [ ]:
def patient_seed_matrix(df, a, b, conditions, regions, metric="dice"):
    """Rows are patients (groups), columns are seeds. Each entry is metric(arm a) minus metric(arm b), averaged
    over the patient's cases, the given conditions and the given regions."""
    sub = df[df.condition.isin(conditions) & df.region.isin(regions) & df.arm.isin([a, b])]
    wide = sub.pivot_table(index=["group", "case_id", "seed", "condition", "region"], columns="arm", values=metric)
    if a not in wide.columns or b not in wide.columns:
        return pd.DataFrame()
    return (wide[a] - wide[b]).groupby(level=["group", "seed"]).mean().unstack("seed")


def boot_ci(values, n_boot, seed, stat=np.mean, alpha=0.05):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return (float("nan"), float("nan"))
    draws = stat(v[np.random.default_rng(seed).integers(0, len(v), (n_boot, len(v)))], axis=1)
    return (float(np.quantile(draws, alpha / 2)), float(np.quantile(draws, 1 - alpha / 2)))


def bca_ci(values, n_boot, seed, alpha=0.05):
    v = np.asarray(values, float)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                res = stats.bootstrap((v,), np.mean, n_resamples=n_boot, confidence_level=1 - alpha, method="BCa",
                                      rng=np.random.default_rng(seed))
            except TypeError:
                res = stats.bootstrap((v,), np.mean, n_resamples=n_boot, confidence_level=1 - alpha, method="BCa",
                                      random_state=np.random.default_rng(seed))
        return (float(res.confidence_interval.low), float(res.confidence_interval.high))
    except Exception:
        return (float("nan"), float("nan"))


def hierarchical_boot_ci(M, n_boot, seed, alpha=0.05):
    """Resample patients (rows) and training seeds (columns) with replacement, so the interval also reflects
    run to run training variability. With three seeds this is coarse; report it next to the patient interval."""
    M = np.asarray(M, float)
    rng = np.random.default_rng(seed)
    n, s = M.shape
    est = np.array([np.nanmean(M[np.ix_(rng.integers(0, n, n), rng.integers(0, s, s))]) for _ in range(n_boot)])
    return (float(np.quantile(est, alpha / 2)), float(np.quantile(est, 1 - alpha / 2)))


def sign_flip_p(values, n_perm, seed):
    v = np.asarray(values, float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(v)))
    null = np.abs((signs * v).mean(1))
    return float((1 + np.count_nonzero(null >= abs(v.mean()) - 1e-12)) / (n_perm + 1))


def blind_labels(df, seed):
    """Swap the M1 and M2 labels within each patient and seed with probability one half. The effect estimate is
    then centered on zero, so the full analysis can be rehearsed without seeing the answer."""
    out = df.copy()
    keys = out[["group", "seed"]].drop_duplicates().reset_index(drop=True)
    keys["swap"] = np.random.default_rng(seed).random(len(keys)) < 0.5
    out = out.merge(keys, on=["group", "seed"], how="left")
    m = out["swap"] & out["arm"].isin(["M1", "M2"])
    out.loc[m, "arm"] = out.loc[m, "arm"].map({"M1": "M2", "M2": "M1"})
    return out.drop(columns="swap")


def confirmatory_analysis(test, cfg, blind=False):
    df = blind_labels(test, cfg.boot_seed) if blind else test
    D = patient_seed_matrix(df, "M2", "M1", MISSING, REGIONS)
    d = D.mean(axis=1).to_numpy()
    Dc = patient_seed_matrix(df, "M2", "M1", ["complete"], REGIONS)
    dc = Dc.mean(axis=1).to_numpy()
    res = {"blind": blind, "n_patients": int(len(d)), "n_seeds": int(D.shape[1]),
           "primary_estimate": float(d.mean()),
           "primary_ci_patient_bootstrap": boot_ci(d, cfg.n_boot, cfg.boot_seed),
           "primary_ci_bca": bca_ci(d, cfg.n_boot, cfg.boot_seed),
           "primary_ci_patients_and_seeds": hierarchical_boot_ci(D.to_numpy(), cfg.n_boot, cfg.boot_seed),
           "primary_per_seed": {str(k): float(v) for k, v in D.mean(axis=0).items()},
           "primary_sign_flip_p": sign_flip_p(d, cfg.n_perm, cfg.boot_seed),
           "complete_estimate": float(dc.mean()),
           "complete_ci_patient_bootstrap": boot_ci(dc, cfg.n_boot, cfg.boot_seed)}
    lo, hi = res["primary_ci_patient_bootstrap"]
    res["H1_consistency_helps_under_missingness"] = "supported" if lo > 0 else ("harmful" if hi < 0 else "unclear")
    ni = res["complete_ci_patient_bootstrap"][0] > -cfg.ni_margin
    res["H2_noninferior_with_complete_inputs"] = ("shown" if ni else "not shown") + (
        "" if lo > 0 else " (descriptive only: the fixed sequence stops when H1 is not supported)")
    return res


def mixed_model_check(test):
    try:
        import statsmodels.formula.api as smf
        sub = test[test.arm.isin(["M1", "M2"]) & test.condition.isin(MISSING)].copy()
        sub["seed"] = sub["seed"].astype(str)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = smf.mixedlm("dice ~ C(arm, Treatment('M1')) + C(condition) + C(region) + C(seed)", sub,
                              groups=sub["group"]).fit()
        key = next(k for k in fit.params.index if "arm" in k and "M2" in k)
        lo, hi = fit.conf_int().loc[key]
        return {"coef_M2_minus_M1": float(fit.params[key]), "ci": (float(lo), float(hi)),
                "note": "random intercept per patient; an approximate cross check, not the primary analysis"}
    except Exception as exc:
        return {"status": f"mixed model could not be fitted ({type(exc).__name__}); rely on the bootstrap results"}


def contrast_table(test, cfg, metric="dice", pairs=(("M1", "M0"), ("M2", "M1"), ("M2", "M0")), stat=np.mean,
                   conditions=CONDITIONS):
    rows = []
    for a, b in pairs:
        for cond in conditions:
            for reg in REGIONS:
                d = patient_seed_matrix(test, a, b, [cond], [reg], metric).mean(axis=1).to_numpy()
                d = d[np.isfinite(d)]
                lo, hi = boot_ci(d, cfg.n_boot, cfg.boot_seed, stat=stat)
                rows.append({"contrast": f"{a} vs {b}", "condition": cond, "region": reg, "metric": metric,
                             "estimate": float(stat(d)) if len(d) else np.nan, "ci_low": lo, "ci_high": hi, "n": len(d)})
    return pd.DataFrame(rows)


def arm_summary(test, metric, cfg, regions=REGIONS):
    g = test[test.region.isin(regions)].groupby(["arm", "condition", "region", "group"])[metric].mean().reset_index()
    rows = []
    for (arm, cond, reg), sub in g.groupby(["arm", "condition", "region"]):
        v = sub[metric].to_numpy()
        v = v[np.isfinite(v)]
        lo, hi = boot_ci(v, cfg.n_boot, cfg.boot_seed)
        rows.append({"arm": arm, "condition": cond, "region": reg, "metric": metric,
                     "mean": float(v.mean()) if len(v) else np.nan, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)


def ece_table(counts):
    rows = []
    for (arm, cond, reg), g in counts.groupby(["arm", "condition", "region"]):
        b = g.groupby("bin")[["n", "sum_p", "sum_y"]].sum()
        b = b[b.n > 0]
        rows.append({"arm": arm, "condition": cond, "region": reg,
                     "ece": float((np.abs(b.sum_p - b.sum_y)).sum() / b.n.sum()) if len(b) else np.nan})
    return pd.DataFrame(rows)


ANALYSIS_FUNCS = [patient_seed_matrix, boot_ci, bca_ci, hierarchical_boot_ci, sign_flip_p, blind_labels,
                  confirmatory_analysis, mixed_model_check, contrast_table, arm_summary, ece_table]


def code_hash(funcs):
    parts = []
    for f in funcs:
        try:
            parts.append(inspect.getsource(f))
        except (OSError, TypeError):
            parts.append(marshal.dumps(f.__code__).hex())
    return sha256_text("\n".join(parts))

In [ ]:
LOCKBOX = DIRS["eval"] / "lockbox.json"
TEST_CSV = DIRS["eval"] / "test_metrics.csv"
TEST_STAMP = DIRS["eval"] / "test_metrics.lockbox_sha256"
CALIB_CSV = DIRS["eval"] / "reliability_counts.csv"


def write_lockbox():
    names = [run_name(a, s, l) for a, s, l in FINAL_RUNS]
    missing = [n for n in names if not (DIRS["runs"] / n / "summary.json").exists()]
    assert not missing, f"unfinished runs: {missing}"
    payload = {"split_sha256": SPLIT_SHA256, "lambda": LAMBDA, "analysis_code_sha256": code_hash(ANALYSIS_FUNCS),
               "checkpoints": {n: sha256_file(DIRS["runs"] / n / "best.pt") for n in names},
               "precision": json.loads(PRECISION_FILE.read_text()), "config": asdict(CFG)}
    if LOCKBOX.exists():
        old = json.loads(LOCKBOX.read_text())
        for k in ("split_sha256", "lambda", "analysis_code_sha256", "checkpoints"):
            if old[k] != payload[k]:
                raise RuntimeError(f"lockbox mismatch in {k}: something changed after the lockbox was written")
        return old
    payload["written"] = stamp()
    LOCKBOX.write_text(json.dumps(payload, indent=1))
    return payload


def evaluate_test():
    if TEST_CSV.exists():
        if not TEST_STAMP.exists() or TEST_STAMP.read_text() != sha256_file(LOCKBOX):
            raise RuntimeError("existing test metrics were produced under a different lockbox")
        print("test metrics already exist; loading them instead of evaluating again")
        return pd.read_csv(TEST_CSV)
    lock = json.loads(LOCKBOX.read_text())
    rng = np.random.default_rng(CFG.boot_seed)
    prob_cases = set(rng.choice(DATA.test_ids, size=min(CFG.n_prob_cases, len(DATA.test_ids)), replace=False).tolist())
    frames, calib_rows = [], []
    for arm, seed, lam in FINAL_RUNS:
        name = run_name(arm, seed, lam)
        if sha256_file(DIRS["runs"] / name / "best.pt") != lock["checkpoints"][name]:
            raise RuntimeError(f"{name}: checkpoint changed after the lockbox was written")
        t0, calib = time.time(), {}
        df = evaluate(load_best(name), DATA.test_ids, DATA, full=True,
                      export_dir=DIRS["eval"] / "nifti" / name if EXPORT_NIFTI else None,
                      prob_cases=prob_cases if seed == CFG.seeds[0] else (), calib=calib, tag=name)
        df.insert(0, "run", name)
        df.insert(1, "arm", arm)
        df.insert(2, "seed", seed)
        frames.append(df)
        for (cond, reg), acc in calib.items():
            calib_rows += [{"arm": arm, "seed": seed, "condition": cond, "region": reg, "bin": k,
                            "n": acc[0, k], "sum_p": acc[1, k], "sum_y": acc[2, k]} for k in range(10)]
        print(f"evaluated {name} on {len(DATA.test_ids)} test cases in {time.time() - t0:.0f} s")
    test = pd.concat(frames, ignore_index=True)
    test["group"] = test.case_id.map(DATA.split.set_index("case_id")["group"])
    test.to_csv(TEST_CSV, index=False)
    pd.DataFrame(calib_rows).to_csv(CALIB_CSV, index=False)
    TEST_STAMP.write_text(sha256_file(LOCKBOX))
    return test


def inference_cost(n_cases=5):
    rows = []
    cases = DATA.test_ids[:n_cases]
    for arm in ARMS:
        name = run_name(arm, CFG.seeds[0], LAMBDA if arm == "M2" else 0.0)
        model = load_best(name)
        vols = [np.load(DIRS["cache"] / c / "img.npy") for c in cases]
        predict_volume(model, volume_to_device(vols[0], DATA.canvas), vols[0].shape[-2:], DATA.canvas)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        times = []
        for vol in vols:
            t0 = time.perf_counter()
            predict_volume(model, volume_to_device(vol, DATA.canvas), vol.shape[-2:], DATA.canvas)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        rows.append({"arm": arm, "parameters": sum(p.numel() for p in model.parameters()),
                     "sec_per_volume_mean": float(np.mean(times)), "sec_per_volume_sd": float(np.std(times, ddof=1)) if len(times) > 1 else 0.0,
                     "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2 ** 30 if DEVICE.type == "cuda" else float("nan")})
    return pd.DataFrame(rows)


if UNLOCK_TEST:
    print(json.dumps({k: v for k, v in write_lockbox().items() if k in ("split_sha256", "lambda", "analysis_code_sha256", "written")}, indent=1))
    TEST = evaluate_test()
    INFER = inference_cost()
    INFER.to_csv(DIRS["analysis"] / "inference_cost.csv", index=False)
    print(INFER.to_string(index=False))
else:
    print("test set locked: finish Stages C to E, read the precision analysis, then set UNLOCK_TEST = True")

## Stage G. Blind rehearsal, confirmatory analysis, diagnostics

Run the analysis first with `UNBLIND = False`. The M1 and M2 labels are randomly swapped within patient and seed, so the estimate sits near zero; check that every table and figure renders and reads the way you intend. Then set `UNBLIND = True` once.

Confirmatory claims, in a fixed sequence that keeps the familywise error at 5%:

1. **H1.** Primary estimand (proposal 3.4): patient level mean Dice difference M2 minus M1 over WT, TC, ET, the four single missing conditions and the three seeds; 95% patient bootstrap interval with 2,000 resamples. Supported if the interval lies above zero.
2. **H2.** Only if H1 is supported: noninferiority of M2 to M1 with complete inputs, shown if the lower limit of the 95% interval lies above minus 0.02.

Everything else (BCa and patients by seeds intervals, per seed estimates, sign flip test, mixed model, secondary contrasts, diagnostics) is supporting evidence and is reported with intervals, not used to claim success.

In [ ]:
if UNLOCK_TEST:
    TEST = pd.read_csv(TEST_CSV)
    blind = confirmatory_analysis(TEST, CFG, blind=True)
    print("BLIND REHEARSAL (labels randomly swapped; numbers carry no information):")
    print(json.dumps(blind, indent=1))
    if UNBLIND:
        RESULT = confirmatory_analysis(TEST, CFG)
        RESULT["mixed_model"] = mixed_model_check(TEST)
        RESULT["lockbox_written"] = json.loads(LOCKBOX.read_text())["written"]
        RESULT["analyzed"] = stamp()
        (DIRS["analysis"] / "confirmatory.json").write_text(json.dumps(RESULT, indent=1))
        print("\nCONFIRMATORY RESULT:")
        print(json.dumps(RESULT, indent=1))

In [ ]:
if UNLOCK_TEST and UNBLIND:
    dice_ct = contrast_table(TEST, CFG)
    hd_ct = contrast_table(TEST, CFG, metric="hd95", stat=np.median)
    pd.concat([dice_ct, hd_ct]).to_csv(DIRS["analysis"] / "secondary_contrasts.csv", index=False)
    print("Dice contrasts (patient level, seeds averaged; secondary, not used to claim success):")
    print(dice_ct.round(4).to_string(index=False))

    # manipulation check: did the consistency term make missing input predictions agree more with complete input ones?
    manip = pd.concat([contrast_table(TEST, CFG, metric=m, pairs=(("M2", "M1"),), conditions=MISSING)
                       for m in ("agree_dice", "kl_to_complete")])
    print("\nManipulation check, M2 vs M1 (agree_dice should rise, kl_to_complete should fall):")
    print(manip.round(4).to_string(index=False))

    TEST["vol_error_ml"] = TEST["pred_ml"] - TEST["gt_ml"]
    diag = pd.concat([arm_summary(TEST, "vol_error_ml", CFG), arm_summary(TEST, "far_fp_ml", CFG, regions=("WT",)),
                      arm_summary(TEST, "brier_roi", CFG)])
    et_absent = TEST[(TEST.region == "ET") & (TEST.gt_ml == 0)]
    fp_rate = (et_absent.assign(fp=et_absent.pred_ml > CFG.min_et_ml).groupby(["arm", "condition"]).fp.mean()
               .unstack("condition") if len(et_absent) else "no ET absent cases in the test set")
    ece = ece_table(pd.read_csv(CALIB_CSV))
    diag.to_csv(DIRS["analysis"] / "diagnostics.csv", index=False)
    ece.to_csv(DIRS["analysis"] / "calibration_ece.csv", index=False)
    print("\nVolume error, far false positives (WT voxels more than "
          f"{CFG.far_fp_mm:g} mm from any reference tumor) and near tumor Brier score:")
    print(diag.round(4).to_string(index=False))
    print(f"\nShare of ET absent test cases with predicted ET above {CFG.min_et_ml} mL "
          f"({et_absent.case_id.nunique()} such cases):")
    print(fp_rate.round(3) if isinstance(fp_rate, pd.DataFrame) else fp_rate)
    print("\nExpected calibration error near tumor:")
    print(ece.pivot_table(index=["arm", "condition"], columns="region", values="ece").round(4))

    f = dice_ct[dice_ct.contrast == "M2 vs M1"].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(6, 0.28 * len(f) + 1))
    yy = np.arange(len(f))[::-1]
    xerr = np.clip(np.vstack([f.estimate - f.ci_low, f.ci_high - f.estimate]), 0, None)
    ax.errorbar(f.estimate, yy, xerr=np.nan_to_num(xerr), fmt="o", ms=3, capsize=2)
    ax.axvline(0, color="grey", lw=0.8)
    ax.axvline(-CFG.ni_margin, color="grey", lw=0.8, ls=":")
    ax.set_yticks(yy)
    ax.set_yticklabels([f"{c}  {r}" for c, r in zip(f.condition, f.region)], fontsize=7)
    ax.set_xlabel("Dice difference, M2 minus M1 (95% patient bootstrap interval)")
    fig.tight_layout()
    fig.savefig(DIRS["analysis"] / "forest_m2_vs_m1.png", dpi=150)
    plt.show()

In [ ]:
if SMOKE_TEST and UNLOCK_TEST:
    et_ids = DATA.split.loc[DATA.split.et_present.astype(bool) & DATA.split.split.isin(["val", "test"]), "case_id"].tolist()
    print(f"Known answer check on the phantom over {len(et_ids)} held out cases with enhancing tumor.")
    print("ET is visible only on t1c. A run that has not learned ET in the short smoke training cannot show the")
    print("pattern and is reported as inconclusive.")
    verdicts = []
    for arm, seed, lam in [r for r in FINAL_RUNS if r[0] != "M0"]:   # M0 never saw missing inputs
        k = evaluate(load_best(run_name(arm, seed, lam)), et_ids, DATA, conditions=("complete", "no_t1c", "no_t1n"))
        s = k[k.region == "ET"].groupby("condition").dice.mean()
        et = s["complete"]
        drop_t1c, drop_t1n = et - s["no_t1c"], et - s["no_t1n"]
        if et < 0.5:
            v = "INCONCLUSIVE"
        else:
            v = "PASS" if drop_t1c > 0.5 and abs(drop_t1n) < 0.15 else "FAIL"
        verdicts.append(v)
        print(f"  {arm} seed {seed}: ET Dice complete {et:.3f}, drop without t1c {drop_t1c:.3f}, "
              f"drop without t1n {drop_t1n:.3f}: {v}")
    if "FAIL" in verdicts:
        print("FAIL: the ET drop appears for the wrong modality; check channel order and modality dropping")
    elif "PASS" in verdicts:
        print("Smoke test finished: every run that learned ET shows the known modality pattern")
    else:
        print("Inconclusive: no run learned ET; increase max_epochs in the smoke configuration and rerun")

## Reporting checklist

1. Report $N_{clean}$, every exclusion with its reason (`audit/exclusions.csv`), how the review flags were resolved, and the split hash.
2. Report the precision analysis (projected interval half width and minimum detectable effect) as it was before the test set was opened, and the lockbox timestamp.
3. Report H1 and H2 exactly as defined above, with the patient bootstrap interval as the primary interval, and the patients by seeds interval and per seed estimates next to it.
4. Report the manipulation check before interpreting H1: if M2 does not raise agreement with the complete input prediction, a null H1 says little about consistency regularization as an idea.
5. Report the T1ce missing condition separately for ET and TC, together with the ET absent false positive rate, since this is where the consistency term can plausibly hurt.
6. Missingness is simulated by removing whole modalities from one random split of one public cohort. The results do not show performance at new hospitals and are not comparable with leaderboard numbers.
7. For an independent check of the metric code, rescore the exported NIfTI predictions in `eval/nifti` with the official BraTS 2023 evaluation code or with MONAI on another machine.